# DOTA v2.0 Oriented Faster R-CNN Benchmark (Google Colab Pro/Pro+)

This notebook establishes a two-track **OBB (`le90`)** benchmark on **DOTA v2.0** (18 classes):
- **Track A (Baseline)**: Rotated Faster R-CNN (`ResNet-50 + FPN`, `1x`, `SmoothL1Loss`)
- **Track B (Flagship)**: Context-Aware Oriented Faster R-CNN (`ConvNeXt-S + PAFPN + Oriented RPN + KLD Loss + ClassBalancedDataset`, `2x`)

Before running:
1. Go to **Runtime -> Change runtime type** -> select **A100 GPU** (or **L4**) + **High-RAM**.
2. Review **Step 2** (`DATA_SOURCE = 'ultralytics'` downloads `DOTAv2.zip` automatically and caches it on Drive).
3. Click **Runtime -> Run all**.


In [ ]:
# Step 1: Mount Google Drive (persistent checkpoints, logs, and raw archive cache)
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi


**Analysis (Step 1):** Verify above that `nvidia-smi` reports an **A100 (40GB/80GB)** or **L4 (24GB)** GPU and that Google Drive is mounted at `/content/drive` for checkpoint persistence.

In [ ]:
# Step 2: Configure Repo, Dataset Source & Run Mode
GITHUB_REPO_URL = 'https://github.com/Lion4467cat/dotav2-oriented-faster-rcnn-benchmark.git'
WORK_ROOT = '/content/drive/MyDrive/dotav2_benchmark'
REPO_DIR = '/content/ADA'
VENV_PY = '/content/mmrotate_venv/bin/python'

# Dataset source: 'huggingface' (full 18-class DOTA-v2.0 1830/593), 'ultralytics', 'local', 'gdrive', or 'kaggle'
DATA_SOURCE = 'huggingface'
KAGGLE_DATASET_SLUG = ''  # Optional if DATA_SOURCE == 'kaggle'
V2_TRAIN_ANN_ID = ''      # Optional if DATA_SOURCE == 'gdrive'
V2_VAL_ANN_ID = ''        # Optional if DATA_SOURCE == 'gdrive'

# Set SMOKE_TEST_LIMIT = 50 for a fast pipeline sanity check, or 0 for full DOTA v2.0
SMOKE_TEST_LIMIT = 0
RUN_BASELINE_TRACK_A = True
RUN_FLAGSHIP_TRACK_B = True


**Analysis (Step 2):** Confirm `DATA_SOURCE` (`'ultralytics'` caches `DOTAv2.zip` in `WORK_ROOT/raw_archives`), and use `SMOKE_TEST_LIMIT = 50` if running an initial pipeline check before full training.

In [ ]:
# Step 3: Clone/Update Repo & Provision Python 3.11 + MMRotate (dev-1.x) via uv
import os, subprocess
os.makedirs(f'{WORK_ROOT}/raw_archives', exist_ok=True)
os.makedirs(f'{WORK_ROOT}/work_dirs', exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', GITHUB_REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

%cd /content/ADA
!bash scripts/setup_colab.sh /content/ADA /content/mmrotate_venv


**Analysis (Step 3):** `setup_colab.sh` creates an isolated Python 3.11 venv via `uv` and installs prebuilt PyTorch 2.1.2 (`cu121`) + MMCV 2.1.0 + MMDet 3.x + MMRotate `dev-1.x`, avoiding Colab's system Python 3.13 incompatibility.

In [ ]:
# Step 4: Download / Extract DOTA v2.0 & Tile 1024x1024 Patches onto Fast Local NVMe
cmd = [
    VENV_PY, 'scripts/download_dotav2.py',
    '--source', DATA_SOURCE,
    '--cache-dir', f'{WORK_ROOT}/raw_archives',
    '--out-dir', '/content/data/dotav2_raw',
]
if KAGGLE_DATASET_SLUG:
    cmd += ['--kaggle-dataset', KAGGLE_DATASET_SLUG]
if V2_TRAIN_ANN_ID:
    cmd += ['--v2-train-ann', V2_TRAIN_ANN_ID]
if V2_VAL_ANN_ID:
    cmd += ['--v2-val-ann', V2_VAL_ANN_ID]
subprocess.run(cmd, check=True)

tile_cmd = [
    VENV_PY, 'scripts/prepare_dotav2_patches.py',
    '--raw-dir', '/content/data/dotav2_raw',
    '--out-dir', '/content/data/split_ss_dotav2',
    '--mmrotate-dir', '/content/ADA/mmrotate',
    '--patch-size', '1024',
    '--overlap', '200',
    '--nproc', '8',
]
if SMOKE_TEST_LIMIT > 0:
    tile_cmd += ['--smoke-test-limit', str(SMOKE_TEST_LIMIT)]
subprocess.run(tile_cmd, check=True)


**Analysis (Step 4):** Inspect the staged instance counts and patch counts printed above. `train` and `val` are tiled independently onto `/content/data/split_ss_dotav2/` to prevent leakage and avoid slow Google Drive FUSE I/O.

In [ ]:
# Step 5: Auto-Tune Batch Size for Detected Colab GPU (A100-80GB / A100-40GB / L4 / T4)
import json
env_info = json.loads(open('/content/ADA/colab_env_summary.json').read())
prof = env_info['recommended_profile']
BS = prof['batch_size']
ACC = prof['accumulative_counts']
print(f"Detected {env_info['gpu_name']} ({env_info['vram_gb']} GB) -> batch_size={BS}, accum={ACC}")


**Analysis (Step 5):** Effective batch size (`BS * ACC = 8`) stays invariant across A100-80GB, A100-40GB, L4, and T4 runtimes so learning rate schedules remain calibrated.

In [ ]:
# Step 6: Train Track A — Vanilla Rotated Faster R-CNN (R50-FPN, 1x Baseline)
if RUN_BASELINE_TRACK_A:
    track_a_dir = f'{WORK_ROOT}/work_dirs/track_a_baseline_r50_fpn_1x'
    !MPLBACKEND=Agg {VENV_PY} mmrotate/tools/train.py \
        configs/baseline_rotated_faster_rcnn_r50_fpn_1x_dotav2.py \
        --work-dir {track_a_dir} \
        --resume \
        --cfg-options train_dataloader.batch_size={BS} optim_wrapper.accumulative_counts={ACC}


**Analysis (Step 6):** Check Track A's validation `dota/mAP` progression and per-class APs in the logs. Checkpoints are persisted to Google Drive every epoch with `--resume` support.

In [ ]:
# Step 7: Train Track B — Context-Aware Oriented Faster R-CNN (ConvNeXt-S + PAFPN + KLD, 2x)
if RUN_FLAGSHIP_TRACK_B:
    track_b_dir = f'{WORK_ROOT}/work_dirs/track_b_convnext_s_pafpn_kld_2x'
    !MPLBACKEND=Agg {VENV_PY} mmrotate/tools/train.py \
        configs/benchmark_oriented_faster_rcnn_convnext_s_pafpn_kld_2x_dotav2.py \
        --work-dir {track_b_dir} \
        --resume \
        --cfg-options train_dataloader.batch_size={BS} optim_wrapper.accumulative_counts={ACC}


**Analysis (Step 7):** Compare Track B's convergence against Track A, paying close attention to rare classes (`helipad`, `airport`, `container-crane`) and high-aspect-ratio classes (`ship`, `bridge`, `large-vehicle`).

In [ ]:
# Step 8: Generate Final Benchmark Report (Saved to Google Drive)
report_cmd = [VENV_PY, 'scripts/evaluate_benchmark.py', '--out', f'{WORK_ROOT}/benchmark_report.md']
if RUN_BASELINE_TRACK_A:
    report_cmd += ['--baseline-dir', f'{WORK_ROOT}/work_dirs/track_a_baseline_r50_fpn_1x']
if RUN_FLAGSHIP_TRACK_B:
    report_cmd += ['--flagship-dir', f'{WORK_ROOT}/work_dirs/track_b_convnext_s_pafpn_kld_2x']
subprocess.run(report_cmd, check=True)


## Conclusion & Benchmark Summary

Review the generated `benchmark_report.md` in Google Drive (`WORK_ROOT/benchmark_report.md`) to inspect:
- Overall `dota/mAP` lift of **Track B (ConvNeXt-S + PAFPN + Oriented RPN + KLD + ClassBalanced)** over **Track A (Rotated Faster R-CNN R50-FPN)**.
- Per-class AP@0.5 across all 18 DOTA v2.0 categories.
- Best checkpoints saved under `WORK_ROOT/work_dirs/*/best_dota_mAP_epoch_*.pth` for downstream inference or `test-dev` submission.
